# Lab: Poses you can chain

You will write the four pieces of pose arithmetic: a pose as a 3×3 matrix, the pose back out of the matrix, the
closed-form inverse, and the step between two poses. Then you chain the 1777 steps of the `gentle` lap back into the
lap itself, break the chain on purpose, and use the same tools to find the car's rear axle.
Read the lecture first; this lab points back to its sections by title and follows its order.

Run the cells from top to bottom (Shift+Enter). Each problem has a "Guess first" line: write your guess down before you
run the cell, then compare. `practice` prints a green tick or tells you what it expected.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/rebuilt_lap.png" alt="The gentle lap seen from above, in metres east and north. In grey, the true path of the car&#x27;s front axle, from the simulator&#x27;s record. In coral, drawn on top, the path rebuilt from nothing but the 1777 steps the car felt, starting from its pose at the tunnel mouth, (0, −1.8), pointing east. Only one line shows, because the two paths are never more than 1.5 trillionths of a metre apart.">
<figcaption><em>Lecture figure 1. The gentle lap seen from above, in metres east and north. In grey, the true path of the car&#x27;s front axle, from the simulator&#x27;s record. In coral, drawn on top, the path rebuilt from nothing but the 1777 steps the car felt, starting from its pose at the tunnel mouth, (0, −1.8), pointing east. Only one line shows, because the two paths are never more than 1.5 trillionths of a metre apart.</em></figcaption>
</figure>

In [ ]:
import sys
if "google.colab" in sys.modules:
    %pip install -q "git+https://github.com/vvknyn/self-driving-ai-course@main"
%matplotlib inline

In [ ]:
import math

import numpy as np

from zero2fsd import geom
from zero2fsd.car import Car, run
from zero2fsd.geom import T_EGO_REAR, rotation_matrix, world_to_ego, wrap_angle
from zero2fsd.grade import check, practice
from zero2fsd.sim import make_scenario
from zero2fsd.sim.scenarios import GENTLE_RADIUS

The next cell drives the `gentle` lap once and keeps the car's pose at each of its 1778 steps: `poses[k]` is
`(x, y, yaw)` in metres and radians. `Tw[k]` is the same pose as a 3×3 matrix, built with the provided
`geom.pose_to_matrix` so that every problem below stands on its own. The lecture used `rotation_matrix` and
`world_to_ego` from *The view from the driver's seat*; swap in your own from that lesson if you like.

In [ ]:
tel = run(Car(), "gentle")
poses = np.column_stack([tel.x, tel.y, tel.yaw])
Tw = np.stack([geom.pose_to_matrix(p) for p in poses])
print(f"{len(poses)} poses; step 400 is x = {poses[400, 0]:.4f}, y = {poses[400, 1]:.4f}, yaw = {poses[400, 2]:.4f} rad")

## 1. A pose is a matrix

The lecture's section "A pose is a frame" puts a pose's axes and origin in the columns of a 3×3 matrix.

**Problem 1.** Write `pose_to_matrix`: from `(x, y, theta)` to the 3×3 matrix whose first two columns are the forward
and left axes and whose last column is `(x, y, 1)`. Use `rotation_matrix(theta)` for the top-left 2×2 block.

**Guess first:** the cone sits 10 m ahead and 2 m to the left of the car at step 400. Will its world position be the
car's position plus (10, 2), that is (164.7, 21.9), or somewhere else?

In [ ]:
def pose_to_matrix(pose):
    """The 3×3 matrix of a pose.

    Input:   pose, a tuple (x, y, theta): metres, metres, radians.
    Output:  a (3, 3) array [[cos, -sin, x], [sin, cos, y], [0, 0, 1]].
    Example: pose_to_matrix((1, 0, 0))  ->  [[1, 0, 1], [0, 1, 0], [0, 0, 1]]
    """
    raise NotImplementedError


try:
    T400 = pose_to_matrix(tuple(poses[400]))
    R400 = rotation_matrix(poses[400, 2])
    practice("the cone (10, 2, trailing 1) at step 400, in world metres; a trailing 0 would leave the car's position out", T400 @ [10, 2, 1], np.append(R400 @ [10, 2] + poses[400, :2], 1), tol=1e-9)
    practice("the arrow (1, 0, trailing 0) at step 400; it must keep only the turn, with no position", T400 @ [1, 0, 0], np.append(R400 @ [1, 0], 0), tol=1e-9)
except NotImplementedError:
    print("Write pose_to_matrix above first, then run this cell again.")

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/pose_frame.png" alt="Two panels stacked. Top: the world frame&#x27;s axes in grey at the world origin, and a long grey arrow from there to the car at step 400, labelled (154.66, 19.85). Bottom: a 20 m square around the car, with its ego frame in purple, the forward axis 40.33° left of east, with a grey dashed line labelled east and an arc for the 40.33° between them. A purple arrow runs 10 m along the forward axis and a second one 2 m along the left axis, ending at the cone, labelled (160.99, 27.85).">
<figcaption><em>Lecture figure 2. Two panels stacked. Top: the world frame&#x27;s axes in grey at the world origin, and a long grey arrow from there to the car at step 400, labelled (154.66, 19.85). Bottom: a 20 m square around the car, with its ego frame in purple, the forward axis 40.33° left of east, with a grey dashed line labelled east and an arc for the 40.33° between them. A purple arrow runs 10 m along the forward axis and a second one 2 m along the left axis, ending at the cone, labelled (160.99, 27.85).</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/pose_matrix.png" alt="The product T times (10, 2, 1) at step 400, drawn as three columns added. 10 times the first column, (0.7623, 0.6473, 0), plus 2 times the second, (−0.6473, 0.7623, 0), plus 1 times the third, (154.6587, 19.8543, 1), equals (160.987, 27.8514, 1). The first column is labelled &quot;forward axis&quot;, the second &quot;left axis&quot; and the third &quot;origin&quot;.">
<figcaption><em>Lecture figure 3. The product T times (10, 2, 1) at step 400, drawn as three columns added. 10 times the first column, (0.7623, 0.6473, 0), plus 2 times the second, (−0.6473, 0.7623, 0), plus 1 times the third, (154.6587, 19.8543, 1), equals (160.987, 27.8514, 1). The first column is labelled &quot;forward axis&quot;, the second &quot;left axis&quot; and the third &quot;origin&quot;.</em></figcaption>
</figure>

The cone is at (160.987, 27.8514): the car's position plus 10 forward axes plus 2 left axes. The arrow has no position
part at all: a trailing 0 skips the third column and keeps only the turn (lecture: "A pose is a frame").

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/arrow_vs_point.png" alt="A 6 m square around the car at step 400, from above, with its ego frame in purple. The ego frame is labelled ego. The same two numbers, (1, 0), sent through T twice. With a trailing 1, (1, 0, 1) comes out as (155.42, 20.50, 1): a purple dot 1 m ahead of the car, a place. With a trailing 0, (1, 0, 0) comes out as (0.7623, 0.6473, 0): an arrow 1 m long pointing the way the car points, drawn in purple at two spots in the square, because an arrow has a direction but no place.">
<figcaption><em>Lecture figure 4. A 6 m square around the car at step 400, from above, with its ego frame in purple. The ego frame is labelled ego. The same two numbers, (1, 0), sent through T twice. With a trailing 1, (1, 0, 1) comes out as (155.42, 20.50, 1): a purple dot 1 m ahead of the car, a place. With a trailing 0, (1, 0, 0) comes out as (0.7623, 0.6473, 0): an arrow 1 m long pointing the way the car points, drawn in purple at two spots in the square, because an arrow has a direction but no place.</em></figcaption>
</figure>

**Break it, fix it.** Send the cone as an arrow, with a trailing 0 instead of a trailing 1. **Guess first:** where does
it land?

In [ ]:
try:
    print("cone with a trailing 0:", np.round(T400 @ [10, 2, 0], 2))
except NameError:
    print("Finish Problem 1 first.")

It lands at (6.33, 8.00): the cone's offset from the car, turned, but with the car's position missing. A trailing 0
turns the numbers and forgets where the car is. To place a point, the last number must be 1.

**Problem 2.** Write `matrix_to_pose`, the way back: `(x, y, theta)` from the matrix. The position is the last column,
and `theta = math.atan2(T[1, 0], T[0, 0])`.

**Guess first:** the car's yaw climbs to 6.2264 rad over the lap. Will `theta` ever be more than π?

In [ ]:
def matrix_to_pose(T):
    """The pose of a 3×3 matrix.

    Input:   T, a (3, 3) array.
    Output:  a tuple (x, y, theta), theta in [-pi, pi].
    Example: matrix_to_pose(pose_to_matrix((1, 2, 0.5)))  ->  (1.0, 2.0, 0.5)
    """
    raise NotImplementedError


try:
    back = np.array([matrix_to_pose(T) for T in Tw])
    practice("worst x or y difference after the round trip, over the lap (m)", np.abs(back[:, :2] - poses[:, :2]).max(), 0.0, tol=1e-12)
    practice("worst theta difference against the wrapped yaw, over the lap (rad); against the raw yaw it fails wherever the yaw is above π", np.abs(back[:, 2] - wrap_angle(poses[:, 2])).max(), 0.0, tol=1e-12)
except NotImplementedError:
    print("Write matrix_to_pose above first, then run this cell again.")

No: `atan2` returns an angle between −π and π, so the round trip gives back the yaw *wrapped* into that range
(lecture: "A pose is a frame", the wrap trap).

**Break it, fix it.** Compare `theta` with the raw yaw instead of the wrapped one. **Guess first:** at which steps will
the raw comparison fail, and by how much?

In [ ]:
try:
    gap = back[:, 2] - poses[:, 2]
    late = np.nonzero(np.abs(gap) > 1)[0]
    above = poses[:, 2] > math.pi
    crossings = np.nonzero(np.diff(above.astype(int)))[0] + 1
    print(f"the raw comparison first fails at step {late[0]}, by {abs(gap[late[0]]):.4f} rad (a full turn); it fails at {len(late)} steps")
    print(f"the yaw crosses π {len(crossings)} times, first at step {crossings[0]} and last at step {crossings[-1]}")
    print("it fails exactly where the yaw is above π:", bool(np.array_equal(late, np.nonzero(above)[0])))
except NameError:
    print("Finish Problem 2 first.")

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/wrap_trap.png" alt="The car&#x27;s yaw over the 88.85 s lap, in radians. In grey, the yaw from the simulator&#x27;s record, which climbs to 6.2264 rad and never wraps. In orange, θ read back by matrix_to_pose, which follows it up to π. At step 1021, 51.05 s in, the orange line first jumps from 3.1412 down to −3.1416. On the back straight the yaw wobbles across π 39 times, as in Positions and headings, so the orange line flips between π and −π until step 1118, 55.90 s in. From there it runs a full turn, 6.2832 rad, below the grey one.">
<figcaption><em>Lecture figure 5. The car&#x27;s yaw over the 88.85 s lap, in radians. In grey, the yaw from the simulator&#x27;s record, which climbs to 6.2264 rad and never wraps. In orange, θ read back by matrix_to_pose, which follows it up to π. At step 1021, 51.05 s in, the orange line first jumps from 3.1412 down to −3.1416. On the back straight the yaw wobbles across π 39 times, as in Positions and headings, so the orange line flips between π and −π until step 1118, 55.90 s in. From there it runs a full turn, 6.2832 rad, below the grey one.</em></figcaption>
</figure>

Wherever the yaw is above π, the two differ by a full turn, 6.2832 rad, though both describe the same direction. The
yaw wobbles across π on the back straight, so the failures start there and then stay. Whenever you compare angles, wrap
the difference. The graders in this lab do.

## 2. Chaining and undoing

"Subscripts that cancel" and "Order matters" are paper-and-pencil sections: write down the product, cancel the inner
names, and check that a swapped product ends somewhere else.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/rear_frame.png" alt="The car at step 400 from above, drawn as a line 2.7 m long from rear axle to front axle. The ego frame sits at the front axle in purple and the rear frame at the rear axle in light purple, both turned 40.33° left of east. The rear axle is labelled (152.60, 18.11).">
<figcaption><em>Lecture figure 6. The car at step 400 from above, drawn as a line 2.7 m long from rear axle to front axle. The ego frame sits at the front axle in purple and the rear frame at the rear axle in light purple, both turned 40.33° left of east. The rear axle is labelled (152.60, 18.11).</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/frame_chain.png" alt="Three frames in a row, drawn as axes: the world frame in grey, the ego frame in purple and the rear frame in light purple, 2.7 m behind the ego frame. Curved arrows run from the rear frame to the ego frame, labelled T ego←rear, and from the ego frame to the world, labelled T world←ego. A longer arrow below runs from the rear frame straight to the world, labelled T world←rear = T world←ego · T ego←rear, with the two ego names struck through. Not to scale.">
<figcaption><em>Lecture figure 7. Three frames in a row, drawn as axes: the world frame in grey, the ego frame in purple and the rear frame in light purple, 2.7 m behind the ego frame. Curved arrows run from the rear frame to the ego frame, labelled T ego←rear, and from the ego frame to the world, labelled T world←ego. A longer arrow below runs from the rear frame straight to the world, labelled T world←rear = T world←ego · T ego←rear, with the two ego names struck through. Not to scale.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/order_matters.png" alt="Two panels stacked, each starting from the same frame at the origin, facing east. Top, A then B: drive 1 m east, turn to face north, drive 1 m north, ending at (1, 1). Bottom, B then A: drive 1 m east, drive 1 m more, then turn to face north, ending at (2, 0). In both the final frame faces north. A faint copy of the other panel&#x27;s end point shows that the two ends are 1.41 m apart.">
<figcaption><em>Lecture figure 8. Two panels stacked, each starting from the same frame at the origin, facing east. Top, A then B: drive 1 m east, turn to face north, drive 1 m north, ending at (1, 1). Bottom, B then A: drive 1 m east, drive 1 m more, then turn to face north, ending at (2, 0). In both the final frame faces north. A faint copy of the other panel&#x27;s end point shows that the two ends are 1.41 m apart.</em></figcaption>
</figure>

**Problem 3.** Write `invert_pose`, the closed-form inverse from "Undoing a pose": `[[Rᵀ, −Rᵀt], [0, 1]]`. It must
work at any size, so take the rotation block as `T[:k, :k]` and the position as `T[:k, k]` with `k = T.shape[0] - 1`.
Do not call `np.linalg.inv`: the formula is cheaper and needs no general solver, and it is the one the next lesson reuses on 4×4
matrices.

**Guess first:** seen from A = (1, 0, π/2), the frame 1 m east of the start and facing north, where is the start?

In [ ]:
def invert_pose(T):
    """The inverse of a rigid transform, in closed form.

    Input:   T, a (k+1, k+1) array: a k×k rotation block, a position column and the bottom row (0, ..., 0, 1).
    Output:  a (k+1, k+1) array, the matrix that undoes T.
    Example: invert_pose(pose_to_matrix((1, 0, math.pi / 2)))  ->  [[0, 1, 0], [-1, 0, 1], [0, 0, 1]]
    """
    raise NotImplementedError


try:
    A = geom.pose_to_matrix((1, 0, math.pi / 2))
    practice("A seen from the start, inverted", invert_pose(A), [[0, 1, 0], [-1, 0, 1], [0, 0, 1]], tol=1e-12)
    practice("the world origin seen from the car at step 400", (invert_pose(Tw[400]) @ [0, 0, 1])[:2], world_to_ego(tuple(poses[400]), (0, 0)), tol=1e-9)
    worst = max(np.abs(invert_pose(T) @ T - np.eye(3)).max() for T in Tw)
    practice("worst entry of invert_pose(T) @ T minus the identity, over the lap", worst, 0.0, tol=1e-12)
    T4 = np.eye(4)
    T4[:3, :3] = geom.rot_z(0.7) @ geom.rot_y(0.1)  # a 3-D turn (next lesson); here it is only a test matrix
    T4[:3, 3] = (1.2, 0.3, 1.5)
    practice("a 4×4 pose: worst entry of invert_pose(T4) @ T4 minus the identity", np.abs(invert_pose(T4) @ T4 - np.eye(4)).max(), 0.0, tol=1e-12)
except NotImplementedError:
    print("Write invert_pose above first, then run this cell again.")

Seen from A, the start is 0 m ahead and 1 m to the left, and it points a quarter turn to A's right. From the car at
step 400 the world origin is at (−130.7431, 84.9692): 130.74 m behind and 84.97 m to the left. The same function
inverts the 4×4 pose, which is why it is written for any size.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/undo_steps.png" alt="Two panels stacked, undoing the cone from the start of this lesson, at step 400. Top, titled take away the position: take the car&#x27;s position away from the cone&#x27;s world point, (160.99, 27.85), and the cone sits at (6.33, 8.00) from the car, still measured along the grey east and north axes. Bottom: turn that back by the car&#x27;s 40.33°, and the cone lands at (10, 2), 10 m along the car&#x27;s purple forward axis and 2 m along its left one.">
<figcaption><em>Lecture figure 9. Two panels stacked, undoing the cone from the start of this lesson, at step 400. Top, titled take away the position: take the car&#x27;s position away from the cone&#x27;s world point, (160.99, 27.85), and the cone sits at (6.33, 8.00) from the car, still measured along the grey east and north axes. Bottom: turn that back by the car&#x27;s 40.33°, and the cone lands at (10, 2), 10 m along the car&#x27;s purple forward axis and 2 m along its left one.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/inverse_A.png" alt="The start frame at the origin in grey, facing east, and the frame A in purple, 1 m east of it and facing north. Seen from A, the start&#x27;s origin is 0 m ahead and 1 m to the left, and the start&#x27;s forward axis points a quarter turn to A&#x27;s right: the pose (0, 1, −π/2).">
<figcaption><em>Lecture figure 10. The start frame at the origin in grey, facing east, and the frame A in purple, 1 m east of it and facing north. Seen from A, the start&#x27;s origin is 0 m ahead and 1 m to the left, and the start&#x27;s forward axis points a quarter turn to A&#x27;s right: the pose (0, 1, −π/2).</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/inverse_view.png" alt="Two panels stacked. Top: the world frame in grey and the car at step 400 in purple, pointing 40.33° left of east, with a grey dot at the world origin, the start of the road&#x27;s centre line. Bottom: the same scene drawn in the car&#x27;s ego frame, the car at the origin facing up the page. The world origin now sits at 130.74 m behind the car and 84.97 m to its left, labelled (−130.74, 84.97).">
<figcaption><em>Lecture figure 11. Two panels stacked. Top: the world frame in grey and the car at step 400 in purple, pointing 40.33° left of east, with a grey dot at the world origin, the start of the road&#x27;s centre line. Bottom: the same scene drawn in the car&#x27;s ego frame, the car at the origin facing up the page. The world origin now sits at 130.74 m behind the car and 84.97 m to its left, labelled (−130.74, 84.97).</em></figcaption>
</figure>

**Break it, fix it.** A tempting slip: use `R` instead of `Rᵀ` in both places. **Guess first:** will the round trip be
slightly off or badly off?

In [ ]:
def invert_pose_slip(T):
    inverse = np.eye(3)
    inverse[:2, :2] = T[:2, :2]
    inverse[:2, 2] = -T[:2, :2] @ T[:2, 2]
    return inverse


slip = [np.abs(invert_pose_slip(T) @ T - np.eye(3)).max() for T in Tw]
print(f"slip: worst entry of the round trip minus the identity is {slip[400]:.3f} at step 400 and {max(slip):.3f} over the lap")

It is off by almost 1 at step 400 and by 2.000 over the lap. The position column cancels; the rotation block turns by
the car's angle twice, 2θ, instead of back to 0, and where θ is near π/2 that is a half turn, so cos 2θ − 1 = −2.
Fix: transpose the rotation block, in both places.

## 3. Odometry: the step between two poses

**Problem 4 (Exercise A).** Write `relative_pose(T_world_a, T_world_b)`, the pose of `b` seen from `a`. In the
lecture's words, `T_a←b = inverse(T_world←a) · T_world←b`. Use your `invert_pose`.

**Guess first:** is `relative_pose(A, A @ B)` equal to `B`? And is `relative_pose(B, B @ A)` equal to `A`?

In [ ]:
def relative_pose(T_world_a, T_world_b):
    """Frame b seen from frame a: the odometry step from pose a to pose b.

    Input:   T_world_a, T_world_b: two pose matrices of one size, (3, 3) on the road and (4, 4) in the next lesson, both in world numbers.
    Output:  an array of the same size, T_a←b: the forward part, the left part and the turn of the move from a to b.
    Example: A = pose_to_matrix((1, 0, math.pi / 2)); B = pose_to_matrix((1, 0, 0))
             relative_pose(A, A @ B)  ->  the matrix of B, [[1, 0, 1], [0, 1, 0], [0, 0, 1]]
    """
    raise NotImplementedError

In [ ]:
check("1.1.4.a", relative_pose)

A correct function passes all 46 cases (five worked ones from the lecture, one pair of 4×4 poses and 40 seeded pairs), every
entry far inside the limit of $10^{-9}$. Now it can turn the car's own record into the steps it felt:

In [ ]:
try:
    steps = np.stack([relative_pose(Tw[k], Tw[k + 1]) for k in range(len(Tw) - 1)])
    practice("mean step forward (m)", steps[:, 0, 2].mean(), 0.3998, tol=0.0001)
    practice("mean step turn (rad)", np.arctan2(steps[:, 1, 0], steps[:, 0, 0]).mean(), 0.00350, tol=0.00001)
except NotImplementedError:
    print("Write relative_pose above first, then run this cell again.")

Each step is about 0.4 m forward and 0.0035 rad of turn on average; the turn is near 0 on the straights.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/one_step.png" alt="One step of the car, step 100 on the first straight, zoomed in, with the angles drawn many times larger than they are. The car&#x27;s frame at step 100 in purple and at step 101 in light purple. The step&#x27;s arrow is split along the step-100 frame&#x27;s axes into 0.4000 m forward and 0.00079 m left, and the turn between the two frames is labelled 0.00027 rad in orange and drawn as an arc between the two forward axes, extended. The same arrow is also split along the grey world axes into 0.39999 m east and 0.00082 m north.">
<figcaption><em>Lecture figure 12. One step of the car, step 100 on the first straight, zoomed in, with the angles drawn many times larger than they are. The car&#x27;s frame at step 100 in purple and at step 101 in light purple. The step&#x27;s arrow is split along the step-100 frame&#x27;s axes into 0.4000 m forward and 0.00079 m left, and the turn between the two frames is labelled 0.00027 rad in orange and drawn as an arc between the two forward axes, extended. The same arrow is also split along the grey world axes into 0.39999 m east and 0.00082 m north.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/step_turns.png" alt="The turn of each of the 1777 odometry steps against time, in radians per step, in orange. On the two straights the turn is about 0. Through each bend it is about 0.005 rad per step: 0.00492 on average through the first bend, and 0.00565 at most. A note says the turns add up to 6.2264 rad.">
<figcaption><em>Lecture figure 13. The turn of each of the 1777 odometry steps against time, in radians per step, in orange. On the two straights the turn is about 0. Through each bend it is about 0.005 rad per step: 0.00492 on average through the first bend, and 0.00565 at most. A note says the turns add up to 6.2264 rad.</em></figcaption>
</figure>

**Break it, fix it.** Swap the arguments: `invert_pose(T_world_b) @ T_world_a`. **Guess first:** what does it say for
`A` and `A @ B`, and does the check notice?

In [ ]:
def relative_pose_swapped(T_world_a, T_world_b):
    return invert_pose(T_world_b) @ T_world_a


try:
    print("swapped, A then B:", np.round(geom.matrix_to_pose(relative_pose_swapped(A, A @ geom.pose_to_matrix((1, 0, 0)))), 4) + 0.0, "(B is (1, 0, 0))")
    check("1.1.4.a", relative_pose_swapped)
except NameError:
    print("Finish Problems 3 and 4 first.")
except NotImplementedError:
    print("Finish Problem 3 first.")

The swapped step comes out as (−1, 0, 0): the step run backwards. The check names the first case it gets wrong. Fix:
invert the *first* pose, `invert_pose(T_world_a) @ T_world_b`.

## 4. Dead reckoning

**Problem 5 (Exercise B).** Write `dead_reckon(T_world_start, steps)`. It takes the start pose and an array of steps,
shape `(n, 3, 3)`, and returns the `n + 1` poses: the start, then each pose times the next step, with the step on the
right (lecture: "Dead reckoning"). The check rebuilds the whole `gentle` lap from its 1777 steps and compares *every*
pose with the simulator's record, not only the last.

In [ ]:
def dead_reckon(T_world_start, steps):
    """The poses a car passes through, rebuilt from the steps it felt.

    Input:   T_world_start: the (3, 3) start pose. steps: an (n, 3, 3) array, steps[k] the step from pose k to pose k + 1.
    Output:  an (n + 1, 3, 3) array: the start, then T_world_{k+1} = T_world_k @ steps[k] for each k.
    Example: dead_reckon(np.eye(3), steps)  with two steps of 1 m forward  ->  poses at x = 0, 1 and 2
    """
    raise NotImplementedError

In [ ]:
check("1.1.4.b", dead_reckon)

A correct function rebuilds all 1778 poses far inside the limits, $10^{-6}$ m and $10^{-9}$ rad. (The check builds its
steps a different way from the lecture, so its worst error is about $5 \times 10^{-13}$ m, not the lecture's
$1.5 \times 10^{-12}$ m.) Add the parts of each step to
x, y and theta instead and the lap becomes a straight line 710.66 m from the truth; the lecture draws it.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/naive_add.png" alt="The gentle lap from above in grey, and in coral the lap rebuilt by adding each step&#x27;s forward part to x, its left part to y and its turn to θ. The coral path is a straight line heading east from the start, ending at (710.49, 16.26), far east of the true lap. A double arrow from the true end to the coral end is labelled 710.66 m, and the north axis is marked.">
<figcaption><em>Lecture figure 14. The gentle lap from above in grey, and in coral the lap rebuilt by adding each step&#x27;s forward part to x, its left part to y and its turn to θ. The coral path is a straight line heading east from the start, ending at (710.49, 16.26), far east of the true lap. A double arrow from the true end to the coral end is labelled 710.66 m, and the north axis is marked.</em></figcaption>
</figure>

**Break it, fix it.** Multiply on the other side: `step @ pose`. **Guess first:** will the check pass?

In [ ]:
def dead_reckon_wrong_order(T_world_start, steps):
    poses_out = [np.asarray(T_world_start, dtype=float)]
    for step in steps:
        poses_out.append(step @ poses_out[-1])
    return np.stack(poses_out)


try:
    wrong = dead_reckon_wrong_order(Tw[0], steps)
    error = np.hypot(wrong[:, 0, 2] - poses[:, 0], wrong[:, 1, 2] - poses[:, 1])
    print(f"wrong order: {error[-1]:.2f} m off at the end, but {error.max():.2f} m off at step {error.argmax()}")
    check("1.1.4.b", dead_reckon_wrong_order)
except NameError:
    print("Finish Problems 3, 4 and 5 first.")
except NotImplementedError:
    print("Finish Problems 3 and 4 first.")

The end is 1.11 m off, which a check of the end alone with a 2 m limit would accept, yet at step 866 the rebuilt car is
204.88 m from the truth. That is why the check looks at every pose. Fix: the step goes on the right,
`pose @ step`.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/wrong_order.png" alt="The gentle lap from above in grey, and in coral the lap rebuilt with the step on the left of each product. The coral path swings far off the true lap. At step 866, 43.30 s in, a coral dot at (−95.12, 179.01) and a grey dot at (108.95, 160.77) are joined by a line labelled 204.88 m. The coral path comes back to end 1.11 m from the true end.">
<figcaption><em>Lecture figure 15. The gentle lap from above in grey, and in coral the lap rebuilt with the step on the left of each product. The coral path swings far off the true lap. At step 866, 43.30 s in, a coral dot at (−95.12, 179.01) and a grey dot at (108.95, 160.77) are joined by a line labelled 204.88 m. The coral path comes back to end 1.11 m from the true end.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/wrong_order_error.png" alt="The wrong-order error against time, in metres, over the 88.85 s lap. It stays under 1 m until 12.20 s, rises to a peak of 204.88 m at 43.30 s and falls back to 1.11 m at the end. A dashed grey line at 2 m marks what a check of the end alone would accept.">
<figcaption><em>Lecture figure 16. The wrong-order error against time, in metres, over the 88.85 s lap. It stays under 1 m until 12.20 s, rises to a peak of 204.88 m at 43.30 s and falls back to 1.11 m at the end. A dashed grey line at 2 m marks what a check of the end alone would accept.</em></figcaption>
</figure>

## 5. Drift

The lecture adds a tiny random error to every step's turn and rebuilds the lap again. Now you do it with your own
`dead_reckon`. The cell draws one error per step from a normal distribution with standard deviation 0.0001 rad,
using seed 0, and turns each step by its error before chaining.

**Guess first:** how far off is the rebuilt car at the end of the lap: millimetres, centimetres or metres?

In [ ]:
try:
    rng = np.random.default_rng(0)
    noisy = steps @ np.stack([geom.pose_to_matrix((0, 0, n)) for n in rng.normal(0, 1e-4, len(steps))])
    drifted = dead_reckon(Tw[0], noisy)
    practice("seed 0: end error (m)", np.hypot(*(drifted[-1, :2, 2] - poses[-1, :2])), 0.688, tol=0.001)
except (NameError, NotImplementedError):
    print("Finish Problems 3, 4 and 5 first.")

Metres: one run of 1777 tiny errors ends 0.688 m from the truth. Change the seed and the number changes; over 500
seeds the RMS end error is 0.608 m (the lecture's measurement).

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/drift_end.png" alt="Two panels stacked, each a 30 m square around the end of the gentle lap, from above. The true path ends at a grey dot. Twenty coral paths, rebuilt with seeds 0 to 19, end around it. Top, σ = 0.0001 rad per step: the coral ends lie within 0.944 m of the true end. Each panel has a 10 m scale bar. Bottom, σ = 0.001 rad per step, on the same scale: they lie within 9.454 m, about ten times as far.">
<figcaption><em>Lecture figure 17. Two panels stacked, each a 30 m square around the end of the gentle lap, from above. The true path ends at a grey dot. Twenty coral paths, rebuilt with seeds 0 to 19, end around it. Top, σ = 0.0001 rad per step: the coral ends lie within 0.944 m of the true end. Each panel has a 10 m scale bar. Bottom, σ = 0.001 rad per step, on the same scale: they lie within 9.454 m, about ten times as far.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/drift_growth.png" alt="Two panels stacked, the position error of the rebuilt car against time over the lap. Top, σ = 0.0001 rad: twenty thin grey curves, one per seed, and a thick coral curve, the RMS over all 500 seeds, which reaches 0.097 m at 12.50 s, 0.447 m at 50 s and a peak of 0.712 m at 74.25 s before easing to 0.608 m at the end. Bottom, σ = 0.001 rad: the same shapes, ten times taller, with the RMS peaking at 7.120 m.">
<figcaption><em>Lecture figure 18. Two panels stacked, the position error of the rebuilt car against time over the lap. Top, σ = 0.0001 rad: twenty thin grey curves, one per seed, and a thick coral curve, the RMS over all 500 seeds, which reaches 0.097 m at 12.50 s, 0.447 m at 50 s and a peak of 0.712 m at 74.25 s before easing to 0.608 m at the end. Bottom, σ = 0.001 rad: the same shapes, ten times taller, with the RMS peaking at 7.120 m.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/drift_laps.png" alt="The RMS position error at the end of each lap against the number of laps driven in the tunnel, 1, 2, 4, 8 and 16, with σ = 0.0001 rad and 500 seeds. The laps axis doubles at each tick. The coral points read 0.608, 0.853, 1.236, 1.737 and 2.454 m and keep rising.">
<figcaption><em>Lecture figure 19. The RMS position error at the end of each lap against the number of laps driven in the tunnel, 1, 2, 4, 8 and 16, with σ = 0.0001 rad and 500 seeds. The laps axis doubles at each tick. The coral points read 0.608, 0.853, 1.236, 1.737 and 2.454 m and keep rising.</em></figcaption>
</figure>

## 6. The rear axle is a frame too

**Worked example 6.** The rear frame is fixed to the car's body, so one product places the rear axle at every step:
`Tw @ T_EGO_REAR`. The code below is given: read it, run it, and compare step 400 with the by-hand answer, the front
position minus 2.7 m along the yaw direction.

**Guess first:** on the first bend, where does the rear axle ride: on the front axle's line, nearer the lane centre,
or further inside?

In [ ]:
rear = Tw @ T_EGO_REAR
by_hand = np.column_stack([poses[:, 0] - 2.7 * np.cos(poses[:, 2]), poses[:, 1] - 2.7 * np.sin(poses[:, 2])])
practice("the rear axle at step 400 (x, y)", rear[400, :2, 2], (152.6006, 18.1067), tol=0.0001)
practice("worst difference from the by-hand answer over the lap (m)", np.abs(rear[:, :2, 2] - by_hand).max(), 0.0, tol=1e-9)

road = make_scenario("gentle").road
s = np.array([road.ego_truth(*p)[0] for p in poses])  # distance along the road
e_front = np.asarray(tel.lat)  # lateral error e at the front axle, from the simulator's record
e_rear = np.array([road.ego_truth(x, y, yaw)[1] for (x, y), yaw in zip(rear[:, :2, 2], poses[:, 2])])
bend = (s > 160) & (s < 100 + math.pi * GENTLE_RADIUS - 60)  # the middle of the first bend
practice("mean lateral error e in the first bend, front axle (m)", e_front[bend].mean(), 0.5298, tol=0.0001)
practice("mean lateral error e in the first bend, rear axle (m)", e_rear[bend].mean(), 0.5746, tol=0.0001)

The rear axle rides further inside the bend: 0.5746 m against 0.5298 m. Its path is shorter than the front's, which
the triangle in the lecture explains.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/rear_axle_paths.png" alt="A stretch of the first bend, straightened out, with the sideways distances drawn 20 times larger, on axes of equal scale. The lane centre is in grey. The front axle&#x27;s path in purple and the rear axle&#x27;s path in light purple, with the car drawn at one step as a bar 2.7 m long from rear axle to front axle, so it runs from the light purple path to the purple one. The bar is stretched sideways by the same 20 times as the paths; its real angle to the lane is 0.95°. The light purple path runs inside the purple one, toward the bend&#x27;s centre.">
<figcaption><em>Lecture figure 20. A stretch of the first bend, straightened out, with the sideways distances drawn 20 times larger, on axes of equal scale. The lane centre is in grey. The front axle&#x27;s path in purple and the rear axle&#x27;s path in light purple, with the car drawn at one step as a bar 2.7 m long from rear axle to front axle, so it runs from the light purple path to the purple one. The bar is stretched sideways by the same 20 times as the paths; its real angle to the lane is 0.95°. The light purple path runs inside the purple one, toward the bend&#x27;s centre.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/rear_axle_error.png" alt="The lateral error e against time over the first 60 s of the lap, for the front axle in blue and the rear axle in a lighter blue, each labelled at its line&#x27;s end. On the first straight the two lines lie together near 0. Through the first bend both sit about half a metre left, the rear one slightly higher. Just after the bend ends, at 44.65 s, the gap between them is largest, 0.123 m, marked with a dot on each line.">
<figcaption><em>Lecture figure 21. The lateral error e against time over the first 60 s of the lap, for the front axle in blue and the rear axle in a lighter blue, each labelled at its line&#x27;s end. On the first straight the two lines lie together near 0. Through the first bend both sit about half a metre left, the rear one slightly higher. Just after the bend ends, at 44.65 s, the gap between them is largest, 0.123 m, marked with a dot on each line.</em></figcaption>
</figure>

## Challenges

The lecture ends with four paper-and-pencil challenge problems on how drift grows with the number of steps, and why
the wrong order ends so near the truth. Their answers are measured with `dead_reckon` and the steps above.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/levers.png" alt="The gentle lap from above in grey. From the car&#x27;s position every 100 steps, a thin grey line runs to the end of the lap, where the error is measured. Each line is one lever: a yaw error at that step turns the end point about the line&#x27;s other end, the place where the car was then.">
<figcaption><em>Lecture figure 22. The gentle lap from above in grey. From the car&#x27;s position every 100 steps, a thin grey line runs to the end of the lap, where the error is measured. Each line is one lever: a yaw error at that step turns the end point about the line&#x27;s other end, the place where the car was then.</em></figcaption>
</figure>

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/drift_loglog.png" alt="Sideways error against the number of steps on straight, σ = 0.0001 rad, on a log scale: each equal step along either axis multiplies the value by 10. Coral dots: the RMS over 500 seeds at N = 10, 20, 50, 100, 200, 400 and 501. A grey line: the law σ ℓ N^{3/2} / √3, with slope 3/2. The dots sit on the line, from 0.00066 m at N = 10 to 0.26579 m at N = 501.">
<figcaption><em>Lecture figure 23. Sideways error against the number of steps on straight, σ = 0.0001 rad, on a log scale: each equal step along either axis multiplies the value by 10. Coral dots: the RMS over 500 seeds at N = 10, 20, 50, 100, 200, 400 and 501. A grey line: the law σ ℓ N^{3/2} / √3, with slope 3/2. The dots sit on the line, from 0.00066 m at N = 10 to 0.26579 m at N = 501.</em></figcaption>
</figure>

## What's next

You can now chain poses, undo them, and turn a record of motion back into a path, and you know what a small error in
every step does to it. The next lesson, *The camera's point of view*, puts the camera in this arithmetic with a 4×4 pose and
`invert_pose` at any size, so that a pixel can become a point on the road.

<figure>
<img src="https://raw.githubusercontent.com/vvknyn/self-driving-ai-course/main/course/1.1.4-poses-you-can-chain/figures/next_lesson.png" alt="The car from the side on flat ground, with heights stretched about three times. The camera sits 1.4 m above the ground at the front axle, tipped 5° down, with its own small set of axes drawn in teal, labelled camera axes, beside the car&#x27;s axes in purple, labelled car axes, and an orange arc marking the 5°. One ray leaves the camera through a pixel, slants down to the road and ends at a question mark.">
<figcaption><em>Lecture figure 24. The car from the side on flat ground, with heights stretched about three times. The camera sits 1.4 m above the ground at the front axle, tipped 5° down, with its own small set of axes drawn in teal, labelled camera axes, beside the car&#x27;s axes in purple, labelled car axes, and an orange arc marking the 5°. One ray leaves the camera through a pixel, slants down to the road and ends at a question mark.</em></figcaption>
</figure>